# TELUGU DEVOTEE — Wan2.1 scene-motion generator

This notebook reads the existing Episode 2 Markdown, extracts each scene's image and motion prompts, and generates silent AI motion clips with Wan2.1 T2V 1.3B. Finished clips are saved to Google Drive and skipped on later runs.

**Free-tier limitations:** a compatible GPU runtime is required; Colab may not offer one. Do not purchase a runtime for this workflow. Start with one short test clip. This notebook does not publish to YouTube.

**If you already ran the older notebook:** after the install cell completes, use Runtime → Restart session (or Restart runtime), then run cells from the top in order. Do not continue with later cells after an error.


In [ ]:
%pip -q uninstall -y torchao
%pip -q install "diffusers==0.35.1" "transformers==4.55.4" accelerate ftfy imageio imageio-ffmpeg "requests==2.32.4"
print("Pinned Wan dependencies installed. If this runtime had earlier import errors, restart the session before running the next cell.")


In [ ]:
import os, re, gc, requests, torch
from pathlib import Path
from google.colab import drive

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
else:
    print("No GPU in this runtime. Scene parsing can still run, but Wan video generation requires a GPU. Try Runtime > Change runtime type > T4 GPU if offered; do not switch to a paid runtime.")

drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/TELUGU_DEVOTEE/episode2')
VIDEO_DIR = BASE / 'scene_videos'
VIDEO_DIR.mkdir(parents=True, exist_ok=True)
MARKDOWN_URL = 'https://raw.githubusercontent.com/Gopi-chand1/telugu-devotee-automation/main/output/episode_package.md'
response = requests.get(MARKDOWN_URL, timeout=60)
response.raise_for_status()
markdown = response.text
print('Downloaded episode Markdown:', len(markdown), 'characters')

scene_parts = re.split(r'(?im)^###\s*SCENE\s+(\d+)\s*(?:\||[—-])[^\n]*\n', markdown)
scenes = {}
for i in range(1, len(scene_parts), 2):
    number = int(scene_parts[i])
    block = scene_parts[i + 1]
    image_match = re.search(r'(?ims)^IMAGE PROMPT:\s*(.*?)(?=^MOTION PROMPT:|\Z)', block)
    motion_match = re.search(r'(?ims)^MOTION PROMPT:\s*(.*?)(?=^(?:DIALOGUE|TELUGU CHARACTER DIALOGUE):|\Z)', block)
    if image_match and image_match.group(1).strip():
        image_prompt = image_match.group(1).strip()
        motion_prompt = motion_match.group(1).strip() if motion_match else ''
        scenes[number] = image_prompt + (('\n\nMotion direction: ' + motion_prompt) if motion_prompt else '')
print('Scene image prompts found:', len(scenes))
print('Scene numbers:', sorted(scenes)[:5], '...', sorted(scenes)[-5:])
if len(scenes) < 1:
    raise RuntimeError('No IMAGE PROMPT blocks found. Check the episode Markdown format.')

# Safe first test: scene 1 only. After reviewing it, change the range for small batches.
START_SCENE = 1
END_SCENE = 1
# Wan frame counts should follow 4*k+1. 41 frames ≈ 2.56 seconds at 16 fps.
NUM_FRAMES = 41
STEPS = 8
FPS = 16
print(f'Planned scenes {START_SCENE}-{END_SCENE}; frames={NUM_FRAMES}; output={VIDEO_DIR}')


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("STOP: Wan2.1 needs a GPU. Choose Runtime > Change runtime type > T4 GPU if available, then restart and run all cells in order. If no free GPU is offered, stop here—do not buy a runtime.")
from diffusers import WanPipeline
from diffusers.utils import export_to_video

model_id = 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers'
pipe = WanPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()
try:
    pipe.vae.enable_tiling()
except Exception:
    pass
print('Wan2.1 loaded. First model download can take several minutes.')


In [ ]:
from IPython.display import Video, display

if 'pipe' not in globals():
    raise RuntimeError("Wan model is not loaded. Resolve the GPU/import error above, restart the session if packages changed, then run cells in order.")
completed, failed = [], []
for number in range(START_SCENE, END_SCENE + 1):
    if number not in scenes:
        print(f'Scene {number}: no image prompt; skipped')
        failed.append(number)
        continue
    output_path = VIDEO_DIR / f'scene_{number:03d}.mp4'
    if output_path.exists() and output_path.stat().st_size > 100_000:
        print(f'Scene {number}: already exists; resume-safe skip')
        completed.append(str(output_path))
        continue
    print(f'Generating scene {number}...')
    try:
        with torch.inference_mode():
            result = pipe(
                prompt=scenes[number],
                negative_prompt='blurry, low quality, distorted face, extra fingers, extra limbs, deformed hands, duplicated people, flicker, jitter, warped body, plastic skin, modern objects, text, watermark, subtitles, logo',
                height=480, width=832, num_frames=NUM_FRAMES,
                num_inference_steps=STEPS, guidance_scale=5.0,
            )
        export_to_video(result.frames[0], str(output_path), fps=FPS)
        if not output_path.exists() or output_path.stat().st_size < 100_000:
            raise RuntimeError('Output MP4 missing or unexpectedly small')
        completed.append(str(output_path))
        print(f'Scene {number}: PASS — {output_path} ({output_path.stat().st_size:,} bytes)')
        display(Video(str(output_path), embed=True, width=480))
        del result
        gc.collect()
        torch.cuda.empty_cache()
    except Exception as exc:
        print(f'Scene {number}: FAILED — {type(exc).__name__}: {exc}')
        failed.append(number)
        gc.collect()
        torch.cuda.empty_cache()

print('\nBatch finished.')
print('Completed:', len(completed), 'Failed/skipped:', failed)
print('Saved in Google Drive:', VIDEO_DIR)
print('No audio muxing and no YouTube upload were performed.')

## After the first test

1. Run the notebook from the top in order. If you see an install/import error, stop and fix it before running later cells.
2. If the first cell installs pinned dependencies in a runtime where the older incompatible packages were already imported, restart the runtime, then run all cells again.
3. If there is no free GPU option, you cannot run Wan2.1 video generation in that session. Wait and try later; do not buy a runtime.
4. Review `scene_001.mp4` for motion quality, faces, hands, flicker, and character continuity.
5. If it succeeds, change `START_SCENE` and `END_SCENE` to a small range (for example, 2–3), then rerun the scene setup cell and generation cell as needed. Existing files are skipped automatically.
6. Keep batches small. The first test is about 2.56 seconds at 16 fps, not a finished 10-second scene or a complete episode.

No YouTube upload or publishing code is included.
